## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import folium
import geopandas as gpd
from folium.plugins import GroupedLayerControl

## Making Map

### Load in data

In [3]:
elkhart_age = pd.read_csv("Elkhart_Age_Data_Transposed.csv") 
joseph_age = pd.read_csv("St_Joseph_Age_Data_Transposed.csv") 
marshall_age = pd.read_csv("Marshall_Age_Data_Transposed.csv") 

In [5]:
elkhart_income = pd.read_csv("Elkhart_MedianIncome_Transposed.csv") 
joseph_income = pd.read_csv("St_Joseph_MedianIncome_Transposed.csv") 
marshall_income = pd.read_csv("Marshall_MedianIncome_Tranposed.csv")

In [6]:
elkhart_pov = pd.read_csv("Elkhart_PovPercent_Transposed.csv") 
joseph_pov = pd.read_csv("St_Joseph_PovPercent_Transposed.csv") 
marshall_pov = pd.read_csv("Marhsall_PovPercent_Transposed.csv") 

In [7]:
elkhart_sex = pd.read_csv("elkhart_sex.csv") 
joseph_sex = pd.read_csv("joseph_sex.csv") 
marshall_sex = pd.read_csv("marshall_sex.csv") 

In [8]:
elkhart_tracts = pd.read_csv("Elkhart_Tracts_Cleaned.csv")
joseph_tracts = pd.read_csv("StJOE_Renamed_Tract_Column.csv") 
marshall_tracts = pd.read_csv("Marshall_Tracts_Cleaned.csv") 

In [9]:
# Clean each poverty dataset
for df, countyfp in [(elkhart_pov, "039"), (joseph_pov, "141"), (marshall_pov, "099")]:
    df["poverty_rate"] = df["Income in the past 12 months below poverty level: "].str.replace("%","").astype(float)
    # Build proper GEOID = state (18) + countyFP + tract
    df["GEOID"] = df["NAME"].apply(lambda x: f"18{countyfp}{int(float(x)*100):06d}")


### Map Features

In [11]:
#Load Indiana tracts (polygons)
gdf = gpd.read_file("tl_2021_18_tract.shp").to_crs(epsg=4326)

#Define county groups
marshall = gdf[gdf["COUNTYFP"] == "099"]   #Marshall County
stj      = gdf[gdf["COUNTYFP"] == "141"]   #St. Joseph County
elkhart  = gdf[gdf["COUNTYFP"] == "039"]   #Elkhart County

#Build lookup dicts for poverty values
marshall_pov_map = dict(zip(marshall_pov["GEOID"], marshall_pov["poverty_rate"]))
stj_pov_map      = dict(zip(joseph_pov["GEOID"], joseph_pov["poverty_rate"]))
elk_pov_map      = dict(zip(elkhart_pov["GEOID"], elkhart_pov["poverty_rate"]))

#Function to style polygons by poverty value
def poverty_style(pov_dict):
    def _style(feature):
        geoid = feature["properties"]["GEOID"]
        val = pov_dict.get(geoid)
        if val is None:
            return {"fillOpacity": 0, "color": "transparent"}
        # Simple YlOrRd-like bins
        if val > 30:
            fill = "#800026"
        elif val > 20:
            fill = "#BD0026"
        elif val > 10:
            fill = "#E31A1C"
        else:
            fill = "#FC4E2A"
        return {"fillColor": fill, "color": "black", "weight": 0.5, "fillOpacity": 0.7}
    return _style

#Create map
m = folium.Map(location=[gdf.geometry.centroid.y.mean(),
                         gdf.geometry.centroid.x.mean()],
               zoom_start=10)

#Outlines
marshall_outline = folium.GeoJson(
    marshall, name="Marshall County",
    style_function=lambda x: {"fillColor": "transparent", "color": "black",
                              "weight": 3, "fillOpacity": 0.5}
).add_to(m)

stj_outline = folium.GeoJson(
    stj, name="St. Joseph County",
    style_function=lambda x: {"fillColor": "transparent", "color": "black",
                              "weight": 3, "fillOpacity": 0.5}
).add_to(m)

elk_outline = folium.GeoJson(
    elkhart, name="Elkhart County",
    style_function=lambda x: {"fillColor": "transparent", "color": "black",
                              "weight": 3, "fillOpacity": 0.5}
).add_to(m)

#Poverty “heatmap” layers (GeoJson styled by poverty)
marshall_poverty = folium.GeoJson(
    marshall, name="Marshall Poverty Heatmap",
    style_function=poverty_style(marshall_pov_map)
).add_to(m)

stj_poverty = folium.GeoJson(
    stj, name="St. Joseph Poverty Heatmap",
    style_function=poverty_style(stj_pov_map)
).add_to(m)

elk_poverty = folium.GeoJson(
    elkhart, name="Elkhart Poverty Heatmap",
    style_function=poverty_style(elk_pov_map)
).add_to(m)

#Grouped Layer Control
GroupedLayerControl(
    groups={
        "Marshall County": [marshall_outline, marshall_poverty],
        "St. Joseph County": [stj_outline, stj_poverty],
        "Elkhart County": [elk_outline, elk_poverty]
    },
    exclusive_groups=False,
    collapsed=True,
).add_to(m)

m.save("map_toggle.html")


/tmp/ipykernel_494/3585765755.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  m = folium.Map(location=[gdf.geometry.centroid.y.mean(),
/tmp/ipykernel_494/3585765755.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf.geometry.centroid.x.mean()],
